<a href="https://colab.research.google.com/github/nmit-1NT23CS128/Social-Media-Comment-Moderation/blob/main/Social_Media_Comment_Moderation_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets torch scikit-learn pandas numpy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#from huggingface_hub import notebook_login
#notebook_login()


In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/train.csv"
df = pd.read_csv(data_path)

df.head()



In [ ]:
toxicity_columns = [
    'toxic',
    'severe_toxic',
    'obscene',
    'threat',
    'insult',
    'identity_hate'
]

df['label'] = df[toxicity_columns].max(axis=1)


In [ ]:
texts = df['comment_text'].fillna("")
labels = df['label']


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)


In [ ]:
print(train_labels.value_counts())


In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')


In [ ]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=128
)


In [ ]:
print(train_encodings.keys())


Buffered data was truncated after reaching the output size limit.

In [13]:
print(train_encodings['input_ids'][0][:10])


[101, 13055, 26568, 2323, 6402, 1999, 11669, 13055, 26568, 2003]


In [14]:
import torch


In [15]:
class ToxicDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

    def __len__(self):
        return len(self.labels)


In [16]:
test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=128
)

train_dataset = ToxicDataset(train_encodings, train_labels)
test_dataset = ToxicDataset(test_encodings, test_labels)

In [17]:
print(len(train_dataset))
print(len(test_dataset))


127656
31915


In [18]:
!pip install sympy==1.14.0 --force-reinstall
from transformers import DistilBertForSequenceClassification, Trainer, TrainingArguments

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: mpmath
    Found existing installation: mpmath 1.3.0
    Uninstalling mpmath-1.3.0:
      Successfully uninstalled mpmath-1.3.0
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0


In [19]:
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [20]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100
)



In [21]:
from transformers import Trainer


In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.099909,0.096383


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.099909,0.096383
2,0.074146,0.106109


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15958, training_loss=0.09388878535531131, metrics={'train_runtime': 3541.0116, 'train_samples_per_second': 72.101, 'train_steps_per_second': 4.507, 'total_flos': 8455129121415168.0, 'train_loss': 0.09388878535531131, 'epoch': 2.0})

In [24]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

accuracy = accuracy_score(test_labels, preds)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, preds, average='binary')

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Accuracy: 0.9692621024596585
Precision: 0.8733091388980534
Recall: 0.8159679408138101
F1 Score: 0.8436653386454184


In [25]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,          # increase epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,          # KEY CHANGE
    eval_strategy="epoch",
    save_strategy="epoch",
    weight_decay=0.01            # regularization
)


In [26]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# Re-initialize tokenizer if it's not defined (due to potential kernel restart)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Re-initialize model if it's not defined (due to potential kernel restart)
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.save_pretrained("toxic_model")
tokenizer.save_pretrained("toxic_model")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('toxic_model/tokenizer_config.json', 'toxic_model/tokenizer.json')

In [27]:
!mkdir BERT_Toxic_Project

In [28]:
!ls

BERT_Toxic_Project  drive  results  sample_data  toxic_model


In [29]:
%cd BERT_Toxic_Project

/content/BERT_Toxic_Project


In [45]:
pip install matplotlib scikit-learn

In [47]:
%%writefile app.py
import streamlit as st
import numpy as np
import torch
import torch.nn.functional as F
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Load model
model = DistilBertForSequenceClassification.from_pretrained("toxic_model")
tokenizer = DistilBertTokenizer.from_pretrained("toxic_model")
model.eval()

# Sidebar navigation
page = st.sidebar.selectbox(
    "Select Page",
    ["Toxic Comment Detector", "Model Performance"]
)

# -------------------------------
# PAGE 1: LIVE PREDICTION
# -------------------------------
if page == "Toxic Comment Detector":

    st.title("🛡 Toxic Comment Detection using BERT")

    def normalize_text(text):
        text = text.lower()
        text = text.replace("ur", "you are")
        text = text.replace("u ", "you ")
        return text

    user_input = st.text_area("Enter Comment")

    if st.button("Predict"):

        processed = normalize_text(user_input)

        inputs = tokenizer(
            processed,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)
            toxic_prob = probs[0][1].item()

        if toxic_prob > 0.7:
            st.error("⚠ Toxic Comment")
        else:
            st.success("✅ Non Toxic Comment")

        st.write("Toxic Probability:", round(toxic_prob,3))
        st.progress(int(toxic_prob*100))


# -------------------------------
# PAGE 2: MODEL DASHBOARD
# -------------------------------
else:

    st.title("📊 Model Performance Dashboard")

    accuracy = 0.951
    precision = 0.763
    recall = 0.754
    f1 = 0.758

    st.metric("Accuracy", accuracy)
    st.metric("Precision", precision)
    st.metric("Recall", recall)
    st.metric("F1 Score", f1)

    # Example confusion matrix
    y_true = [0,0,0,1,1,1]
    y_pred = [0,0,1,1,1,0]

    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots()
    ax.matshow(cm)
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, val, ha='center', va='center')
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    st.pyplot(fig)

Overwriting app.py


In [31]:
!ls

app.py


In [32]:
model.save_pretrained("toxic_model")
tokenizer.save_pretrained("toxic_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('toxic_model/tokenizer_config.json', 'toxic_model/tokenizer.json')

In [37]:
!pip install pyngrok

In [39]:
!pip install pyngrok streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 74.2 MB/s eta 0:00:00


In [40]:
from pyngrok import ngrok

ngrok.set_auth_token("3BDTF8NoUigVbuJIvmmnidQzfQm_4UTY1Ysq7tMr8ihd3iAC9")

In [41]:
!streamlit run app.py &>/content/logs.txt &

In [42]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://electrophysiological-windiest-kathryne.ngrok-free.dev" -> "http://localhost:8501"
